[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_rnaseq.ipynb) [![Open In nbviewer](https://img.shields.io/badge/View%20in-nbviewer-orange)](https://nbviewer.jupyter.org/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_rnaseq.ipynb)

# Bulk RNA-Seq

This tutorial is a brief guide for the implementation of BiT Age, a highly accurate bulk transcriptomic clock for C. elegans. Link to [paper](https://onlinelibrary.wiley.com/doi/full/10.1111/acel.13320).

We just need two packages for this tutorial.

In [1]:
import pandas as pd
import pyaging as pya

## Download and load example data

Let's download the C. elegans RNA-seq dataset from the BiT Age paper.

In [2]:
pya.data.download_example_data('GSE65765')

⏺ example data already at pyaging_data/GSE65765_CPM.pkl

'pyaging_data/GSE65765_CPM.pkl'

In [3]:
df = pd.read_pickle('pyaging_data/GSE65765_CPM.pkl')

In [4]:
df.head()

,WBGene00197333,WBGene00198386,WBGene00015153,WBGene00002061,WBGene00255704,WBGene00235314,WBGene00001177,WBGene00169236,WBGene00219784,WBGene00015152,...,WBGene00010964,WBGene00014467,WBGene00014468,WBGene00014469,WBGene00014470,WBGene00010965,WBGene00014471,WBGene00010966,WBGene00010967,WBGene00014473
SRR1793993,0.0,0.0,3.780174,169.240815,1.907427,0.277444,59.320986,0.0,0.000000,1.283178,...,858.949156,0.0,0.000000,0.0,0.052021,234.526846,0.017340,54.483057,78.117815,0.000000
SRR1793991,0.0,0.0,0.510354,412.628597,0.061861,0.061861,22.239044,0.0,0.015465,0.201048,...,1049.982885,0.0,0.015465,0.0,0.015465,372.511713,0.000000,54.545971,59.618577,0.000000
SRR1793994,0.0,0.0,4.718708,274.733671,1.234644,0.118391,42.400721,0.0,0.000000,0.642691,...,664.255412,0.0,0.101478,0.0,0.000000,253.220421,0.033826,19.483698,86.492735,0.016913
SRR1793992,0.0,0.0,2.389905,351.612558,0.505892,0.069778,20.497358,0.0,0.017445,1.308342,...,1298.799849,0.0,0.034889,0.0,0.000000,472.206803,0.000000,89.508039,76.459508,0.000000


## Convert data to AnnData object

AnnData objects are highly flexible and are thus our preferred method of organizing data for age prediction.

In [5]:
adata = pya.preprocess.df_to_adata(df)

Note that the original DataFrame is stored in `X_original` under layers. is This is what the `adata` object looks like:

In [6]:
adata

AnnData object with n_obs × n_vars = 4 × 46755
    var: 'percent_na'
    layers: 'X_original', None (.X)

## Predict age

We can either predict one clock at once or all at the same time. Given we only have one clock of interest for this tutorial, let's go with one. The function is invariant to the capitalization of the clock name. 

In [7]:
pya.pred.predict_age(adata, 'BiTAge')

In [8]:
adata.obs.head()

,bitage
SRR1793993,182.353658
SRR1793991,27.337245
SRR1793994,241.629584
SRR1793992,32.178003


After age prediction, the clocks are added to `adata.obs`. Moreover, the percent of missing values for each clock and other metadata are included in `adata.uns`.

In [9]:
adata

AnnData object with n_obs × n_vars = 4 × 46755
    obs: 'bitage'
    var: 'percent_na'
    uns: 'bitage_percent_na', 'bitage_missing_features', 'bitage_metadata'
    layers: 'X_original', None (.X)

## Get citation

The doi, citation, and some metadata are automatically added to the AnnData object under `adata.uns[CLOCKNAME_metadata]`.

In [10]:
adata.uns['bitage_metadata']

{'clock_name': 'bitage',
 'data_type': 'transcriptomics',
 'species': 'Caenorhabditis elegans',
 'year': 2021,
 'approved_by_author': '✅',
 'citation': 'Meyer, David H., and Björn Schumacher. "BiT age: A transcriptome-based aging clock near the theoretical limit of accuracy." Aging Cell 20 (2021): e13320.',
 'doi': 'https://doi.org/10.1111/acel.13320',
 'notes': 'Binarized whole-organism C. elegans RNA-seq clock that estimates temporally rescaled biological age; the released linear predictor sums coefficients for genes binarized on plus a 103.55-hour intercept.',
 'research_only': None,
 'tissue': ['whole organism'],
 'predicts': ['biological age'],
 'training_target': ['biological age'],
 'unit': ['hours'],
 'model_type': 'elastic net regression',
 'platform': ['RNA-seq'],
 'population': 'Caenorhabditis elegans',
 'journal': 'Aging Cell',
 'last_author': 'Björn Schumacher',
 'n_features': 576,
 'citations': 173,
 'citations_date': '2026-07-05',
 'version': 'v0.3.0',
 'preprocess': 'bi

## Cohort-relative clocks: tAge

`tage` and `tagemortality` come from Tyshkovskiy, Alexander, et al. "Universal transcriptomic hallmarks of mammalian ageing and mortality." *Nature* 654 (2026): 173-188. Link to [paper](https://doi.org/10.1038/s41586-026-10542-3). They are released under the MGB Open Access License 1.0 — **non-commercial academic research use only**, which is why the Clock Catalogue marks them "Research use only".

They cover mouse, rat, macaque, and human, so the C. elegans cohort above is not an input for them; the examples below assume your own bulk RNA-seq counts from one of those four species. Two things are worth understanding before running them.

**They read a cohort, not a sample.** Normalisation and centring are estimated across every sample in the call, so a sample's prediction depends on the samples predicted alongside it — the same sample scored in a different cohort gets a different number. Two samples is the hard minimum; below roughly ten the statistics are too noisy to read much into. `predict_age` runs this preprocessing itself on raw counts, so there is nothing to call from `pyaging.preprocess` first, and `adata.X` is left untouched.

**The prediction is a difference, not an age.** `tage` returns months of *mouse* age relative to the reference group: zero means "looks like the reference", negative means younger-looking. Mouse months are the unit whatever the species, because the model predicts a fraction of maximum lifespan and the mouse factor of 48 months is baked into the weights — rescale by your species' maximum lifespan over 48 (human 122.5 years, rat 50.4 months, macaque 39 years) to read it on its own timescale. `tagemortality` returns a log hazard ratio against the same reference.

This section is written as code blocks rather than run cells: `tage` and `tagemortality` are not yet published to the pyaging Hugging Face repositories, so `predict_age` has no weights to fetch. They ship with the v0.5.x release, at which point these blocks run as written against your own cohort.

### A default run

The input is a raw count matrix, samples by genes, indexed by gene identifiers of your cohort's own species — symbols, Ensembl, or Entrez IDs all work. The models are defined over mouse Entrez IDs and pyaging maps your identifiers onto them, so a human cohort is labelled with human genes, not translated by hand; anything the models expect but your data does not measure falls back to its training median. Sample annotations can ride along in the same frame: `metadata_cols` moves those columns into `.obs` instead of reading them as genes, and the `treatment` column below is what the reference-group example further down selects on.

```python
mouse_adata = pya.preprocess.df_to_adata(counts, metadata_cols=['treatment'])
pya.pred.predict_age(mouse_adata, ['tage', 'tagemortality'])
mouse_adata.obs[['tage', 'tagemortality']]
```

With no reference group named, the cohort centres on itself, so each prediction says how that sample's expression compares with the cohort as a whole. Note that it is the expression that is centred, not the predictions: the cohort's mean prediction is near zero but not exactly zero.

### Naming the species

The run above never said which species the cohort is. With no species column pyaging assumes mouse and logs a warning — and that warning is the only signal, so `verbose=False` makes the assumption silent. Name the species explicitly in anything scripted.

The idiom is a column set to `1` for every sample, the same one the mammalian methylation clocks use for covariates such as `female`. Valid names are `mouse`, `rat`, `macaque`, and `human`, matched case-insensitively. For a human cohort:

```python
human_counts['human'] = 1
human_adata = pya.preprocess.df_to_adata(human_counts, metadata_cols=['treatment'])
pya.pred.predict_age(human_adata, ['tage', 'tagemortality'])
```

Exactly one may be set, and the column is dropped before the gene pipeline rather than read as a gene — which is why it goes in the count frame rather than in `metadata_cols`. Setting two, or letting one vary between samples, is an error rather than a guess.

### Choosing the reference group

Centring on the whole cohort answers "which of these samples look older than the others". Usually the more useful question is "how do the treated animals look next to the controls", which means centring on a subset. Mark it with a truthy `adata.obs['tage_reference_group']`, boolean or `0`/`1`.

```python
mouse_adata.obs['tage_reference_group'] = mouse_adata.obs['treatment'] == 'control'
pya.pred.predict_age(mouse_adata, ['tage', 'tagemortality'])
```

Changing the reference group shifts every prediction by the same amount and leaves the spread between samples alone — what moves is the baseline the deviations are measured from, now the controls rather than the cohort mean. So it is differences that carry the meaning: a treated animal scoring 3.2 mouse-months below the control average is 3.2 months of mouse age younger-looking than those controls. A column that selects no samples raises rather than quietly falling back to the whole cohort.

What the preprocessing actually did — the species it used, how many genes mapped, how large the reference group was — is recorded in `mouse_adata.uns['tage_preparation']`, and the usual `adata.uns['tage_metadata']` carries the citation and licence.